# 01 — Data Dictionary
This notebook loads the complete project datasets for the configured period and generates `reports/data_dictionary.csv` and `docs/Data_Dictionary.md`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
START_YEAR = 2021
END_YEAR = 2025

PROJECT_ROOT, PROCESSED_DIR

(WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk'),
 WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/data/processed'))

In [2]:
from src.ontario_peak_risk.utils.io import load_project_datasets

datasets = load_project_datasets(
    PROCESSED_DIR,
    start_year=START_YEAR,
    end_year=END_YEAR,
)

{
    'consumption_shape': datasets.consumption.shape,
    'weather_shape': datasets.weather.shape,
    'calendar_shape': datasets.calendar.shape,
}

{'consumption_shape': (1046461, 12),
 'weather_shape': (87648, 34),
 'calendar_shape': (43824, 46)}

In [5]:
from src.ontario_peak_risk.data_dictionary.build_data_dictionary import build_data_dictionary

data_dictionary = build_data_dictionary(
    processed_directory=PROCESSED_DIR,
    project_root=PROJECT_ROOT,
    start_year=START_YEAR,
    end_year=END_YEAR,
)

data_dictionary.head(20)

Data dictionary CSV: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\reports\data_dictionary.csv
Data dictionary Markdown: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\docs\data_dictionary.md
Fields documented: 89


,dataset,field_name,description,source_dtype_observed,unit,analytical_role,nullable_observed,non_null_count,missing_count,missing_pct,unique_count,example_values,source_reference,notes
0,consumption,FSA,Forward Sortation Area: the first three charac...,string,,Geographic identifier / grouping variable,False,1046461,0,0.0,6,L4T | M5R | M5S,Independent Electricity System Operator. Hourl...,
1,consumption,DATE,Source date associated with the IESO hourly re...,datetime64[ns],,Temporal identifier,False,1046461,0,0.0,1826,2021-01-01 | 2021-01-02 | 2021-01-03,Independent Electricity System Operator. Hourl...,Interpret together with HOUR and the documente...
2,consumption,HOUR,"IESO hour-ending number, represented from 1 th...",int64,,Temporal identifier,False,1046461,0,0.0,24,1 | 2 | 3,Independent Electricity System Operator. Hourl...,
3,consumption,CUSTOMER_TYPE,Customer category used by IESO for the aggrega...,string,,Segmentation dimension,False,1046461,0,0.0,2,Residential | SGS <50kW,Independent Electricity System Operator. Hourl...,
4,consumption,PRICE_PLAN,Electricity price-plan category associated wit...,string,,Segmentation dimension,False,1046461,0,0.0,4,Retailer | TOU | Tiered,Independent Electricity System Operator. Hourl...,
5,consumption,TOTAL_CONSUMPTION,Aggregated electricity consumption for the pre...,float64,kWh,Primary demand measure / future forecasting ta...,False,1046461,0,0.0,118781,177.5 | 7316.3 | 112.1,Independent Electricity System Operator. Hourl...,
6,consumption,PREMISE_COUNT,Number of premises included in the aggregated ...,int64,premises,Coverage and normalization measure,False,1046461,0,0.0,2389,194 | 7442 | 154,Independent Electricity System Operator. Hourl...,It is not a population count and may reflect s...
7,consumption,SOURCE_PERIOD,Monthly source period extracted during preproc...,int64,,Traceability field,False,1046461,0,0.0,60,202101 | 202102 | 202103,Project-generated variable. Definition based o...,Expected format: YYYYMM.
8,consumption,INTERVAL_START_TIMESTAMP,Start of the hourly consumption interval deriv...,datetime64[ns],,Primary temporal integration key,False,1046461,0,0.0,43824,2021-01-01 00:00:00 | 2021-01-01 01:00:00 | 20...,Project-generated variable. Definition based o...,
9,consumption,INTERVAL_END_TIMESTAMP,End of the hourly consumption interval derived...,datetime64[ns],,Temporal audit field,False,1046461,0,0.0,43824,2021-01-01 01:00:00 | 2021-01-01 02:00:00 | 20...,Project-generated variable. Definition based o...,


In [4]:
data_dictionary.groupby('dataset').agg(
    fields=('field_name', 'count'),
    fields_with_missing=('missing_count', lambda s: int((s > 0).sum())),
    fields_requiring_manual_review=(
        'source_reference',
        lambda s: int(s.astype(str).str.contains('Manual verification', case=False).sum()),
    ),
)

,fields,fields_with_missing,fields_requiring_manual_review
dataset,,,
calendar,45,1,0
consumption,11,0,0
weather,33,22,0


## Manual review
Open `docs/Data_Dictionary.md` and review any field marked **Manual verification required**. Do not replace those notes with assumptions.